# PaddleOCR - Training sur `batch_1` + validation qualitative sur `batch_2`

Ce notebook:
- telecharge le dataset Kaggle
- convertit les annotations CSV de `batch_1` en format PaddleOCR
- cree un split interne train/val depuis `batch_1` (supervise)
- entraine des modeles `det` et `rec`
- collecte et trace les metriques d'entrainement par epoch
- exporte les modeles d'inference
- lance une inference brute sur un JPG cible
- lance une validation qualitative non annotee sur les images de `batch_2`

Important: `batch_2` est traite comme jeu non annote (pas de metriques GT).

Note: `val_loss` n'est pas toujours expose par les logs standards PaddleOCR; le notebook trace donc aussi les metriques de validation disponibles et un CER approxime pour la reco.

In [ ]:
# Dependances (a executer une fois si necessaire)
# Si Paddle n'est pas deja installe dans ton environnement, choisis UNE option:
# 1) GPU (adapter selon ta stack CUDA)
# !pip install -q paddlepaddle-gpu==2.6.2.post120 -f https://www.paddlepaddle.org.cn/whl/linux/mkl/avx/stable.html
# 2) CPU
# !pip install -q paddlepaddle==2.6.2

# Dependances projet
# !pip install -q kagglehub pandas numpy opencv-python pyyaml matplotlib tqdm paddleocr

In [ ]:
import ast
import json
import math
import os
import random
import re
import subprocess
import sys
from collections import defaultdict
from pathlib import Path

import cv2
import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import paddle
import yaml
from IPython.display import display
from tqdm.auto import tqdm

print(f"Python: {sys.version.split()[0]}")
print(f"Paddle: {paddle.__version__}")
print(f"CUDA available: {paddle.is_compiled_with_cuda()}")
if paddle.is_compiled_with_cuda():
    print(f"CUDA devices: {paddle.device.cuda.device_count()}")

In [ ]:
# Configuration
DATASET_SLUG = "osamahosamabdellatif/high-quality-invoice-images-for-ocr"
TRAIN_BATCH_NAME = "batch_1"
UNLABELED_VAL_BATCH_NAME = "batch_2"

TRAIN_VAL_SPLIT = 0.9  # split supervise interne sur batch_1
RANDOM_SEED = 42
BATCH2_MAX_IMAGES = 300  # limite de validation qualitative sur batch_2

WORK_DIR = Path("./workspace_paddleocr_invoice").resolve()
PREP_DIR = WORK_DIR / "prepared_data"
RUNS_DIR = WORK_DIR / "runs"
EXPORT_DIR = WORK_DIR / "export"

# Parametres d'entrainement (ajuste selon ton GPU/temps)
DET_EPOCHS = 50
REC_EPOCHS = 50
USE_GPU = paddle.is_compiled_with_cuda() and paddle.device.cuda.device_count() > 0

TEST_IMAGE = Path("/mnt/c/Users/fback/Downloads/batch3-0501.jpg")

for p in [WORK_DIR, PREP_DIR, RUNS_DIR, EXPORT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print("WORK_DIR:", WORK_DIR)
print("USE_GPU:", USE_GPU)

In [ ]:
# Profil d'entrainement adapte a la VRAM (version Paddle)
def detect_training_profile(use_gpu: bool):
    cpu_count = os.cpu_count() or 2
    workers = max(2, min(8, cpu_count // 2))

    profile = {
        "device": "gpu" if use_gpu else "cpu",
        "gpu_name": None,
        "vram_gb": 0.0,
        "train_batch_size": 2,
        "eval_batch_size": 2,
        "gradient_accumulation_steps_requested": 8,
        "effective_batch_target": 16,
        "num_workers": min(4, workers),
        "eval_num_workers": min(4, workers),
        "use_amp": False,
        "amp_level": "O2",
        "scale_loss": 1024.0,
        "det_eval_batch_size": 1,  # det eval stable: 1
    }

    if use_gpu:
        props = paddle.device.cuda.get_device_properties()
        vram_gb = float(props.total_memory) / (1024 ** 3)

        profile.update(
            {
                "gpu_name": str(props.name),
                "vram_gb": round(vram_gb, 1),
                "use_amp": True,
                "num_workers": workers,
                "eval_num_workers": workers,
            }
        )

        if vram_gb >= 14:
            train_bs, eval_bs, accum = 8, 8, 2
        elif vram_gb >= 10:
            train_bs, eval_bs, accum = 6, 6, 2
        elif vram_gb >= 7.5:
            train_bs, eval_bs, accum = 4, 4, 4
        else:
            train_bs, eval_bs, accum = 2, 2, 8

        profile.update(
            {
                "train_batch_size": train_bs,
                "eval_batch_size": eval_bs,
                "gradient_accumulation_steps_requested": accum,
                "effective_batch_target": train_bs * accum,
            }
        )

    return profile


TRAINING_PROFILE = detect_training_profile(USE_GPU)

print("Profil d'entrainement detecte")
for key, value in TRAINING_PROFILE.items():
    print(f"  {key}: {value}")

if TRAINING_PROFILE["gradient_accumulation_steps_requested"] > 1:
    print("Note: gradient accumulation est une cible informative ici (non expose nativement par train.py PaddleOCR).")


In [ ]:
# Download dataset
dataset_path = Path(kagglehub.dataset_download(DATASET_SLUG)).resolve()
print("Dataset path:", dataset_path)

def print_tree(root: Path, max_depth: int = 3, max_entries: int = 120):
    root = root.resolve()
    count = 0
    for p in sorted(root.rglob("*")):
        depth = len(p.relative_to(root).parts)
        if depth > max_depth:
            continue
        print("  " * (depth - 1) + ("- " if depth > 0 else "") + p.name)
        count += 1
        if count >= max_entries:
            print("... (truncated)")
            break

print_tree(dataset_path, max_depth=3, max_entries=120)

In [ ]:
# Helpers CSV -> format PaddleOCR (support labels documentaires sans bbox)
from difflib import SequenceMatcher

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

def norm_col(s: str) -> str:
    return re.sub(r"[^a-z0-9]", "", str(s).lower())

def find_col(cols, candidates):
    nmap = {c: norm_col(c) for c in cols}
    for cand in candidates:
        ncand = norm_col(cand)
        for c, n in nmap.items():
            if n == ncand:
                return c
    for cand in candidates:
        ncand = norm_col(cand)
        for c, n in nmap.items():
            if ncand in n:
                return c
    return None

def find_batch_root(root: Path, batch_name: str) -> Path:
    candidates = [p for p in root.rglob(batch_name) if p.is_dir()]
    if not candidates:
        raise FileNotFoundError(f"Impossible de trouver le dossier '{batch_name}' sous {root}")
    scored = []
    for p in candidates:
        csv_count = len(list(p.glob("*.csv")))
        img_count = sum(1 for f in p.rglob("*") if f.is_file() and f.suffix.lower() in IMAGE_EXTS)
        scored.append((csv_count, img_count, -len(p.parts), p))
    scored.sort(reverse=True)
    return scored[0][-1]

def build_image_lookup(batch_root: Path):
    lookup = {}
    for p in batch_root.rglob("*"):
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
            rel = str(p.relative_to(batch_root)).replace("\\", "/").lower()
            lookup.setdefault(rel, p)
            lookup.setdefault(p.name.lower(), p)
    return lookup

def list_images_in_batch(dataset_root: Path, batch_name: str):
    batch_root = find_batch_root(dataset_root, batch_name)
    images = sorted([p for p in batch_root.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTS])
    return batch_root, images

def resolve_image_path(raw_value, batch_root: Path, image_lookup):
    if pd.isna(raw_value):
        return None
    raw = str(raw_value).strip()
    if not raw:
        return None

    candidates = [raw, raw.replace("\\", "/"), raw.lstrip("./"), Path(raw).name]
    for c in candidates:
        if c.lower() in image_lookup:
            return image_lookup[c.lower()]
    for c in candidates:
        p = (batch_root / c).resolve()
        if p.exists() and p.suffix.lower() in IMAGE_EXTS:
            return p
    stem = Path(raw).stem.lower()
    for _, p in image_lookup.items():
        if p.stem.lower() == stem:
            return p
    return None

def load_batch_document_labels(dataset_root: Path, batch_name: str):
    batch_root = find_batch_root(dataset_root, batch_name)
    csv_files = sorted(batch_root.glob("*.csv"))
    if not csv_files:
        raise FileNotFoundError(f"Aucun CSV trouve dans {batch_root}")

    image_lookup = build_image_lookup(batch_root)

    image_col_candidates = ["File Name", "file_name", "filename", "image", "image_path", "img", "path"]
    text_col_candidates = ["OCRed Text", "ocr_text", "text", "label", "transcription", "content"]

    doc_labels = {}
    report = []

    for csv_path in csv_files:
        df = pd.read_csv(csv_path)
        cols = list(df.columns)
        image_col = find_col(cols, image_col_candidates)
        text_col = find_col(cols, text_col_candidates)

        if image_col is None or text_col is None:
            raise ValueError(
                f"CSV incompatible: {csv_path.name}\n"
                f"Colonnes trouvees: {cols}\n"
                f"Image col: {image_col} | Text col: {text_col}"
            )

        kept = 0
        for _, row in df.iterrows():
            img_path = resolve_image_path(row.get(image_col), batch_root, image_lookup)
            if img_path is None:
                continue
            text = str(row.get(text_col, "")).strip()
            if not text:
                continue

            key = str(img_path)
            prev = doc_labels.get(key)
            if prev is None or len(text) > len(prev):
                doc_labels[key] = text
            kept += 1

        report.append({"csv": str(csv_path), "rows": len(df), "kept_rows": kept, "columns": cols})

    return batch_root, doc_labels, report

def split_document_labels(doc_labels: dict, train_ratio: float, seed: int):
    keys = sorted(doc_labels.keys())
    if len(keys) == 0:
        return {}, {}
    if len(keys) == 1:
        only = keys[0]
        return {only: doc_labels[only]}, {only: doc_labels[only]}

    rng = np.random.default_rng(seed)
    rng.shuffle(keys)

    train_n = int(round(len(keys) * train_ratio))
    train_n = max(1, min(len(keys) - 1, train_n))

    train_keys = keys[:train_n]
    val_keys = keys[train_n:]

    train_docs = {k: doc_labels[k] for k in train_keys}
    val_docs = {k: doc_labels[k] for k in val_keys}
    return train_docs, val_docs

def doc_similarity(a: str, b: str) -> float:
    a = (a or "").strip().lower()
    b = (b or "").strip().lower()
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    return float(SequenceMatcher(None, a, b).ratio())

def build_pseudo_annotations_from_teacher(doc_labels: dict, use_gpu: bool, lang: str = "en"):
    from paddleocr import PaddleOCR

    teacher = PaddleOCR(
        use_angle_cls=True,
        lang=lang,
        device=("gpu:0" if use_gpu else "cpu"),
    )

    by_image = {}
    quality_rows = []

    for img_path, gt_text in tqdm(doc_labels.items(), desc="Pseudo-labeling with teacher OCR"):
        result = teacher.ocr(str(img_path), cls=True)

        anns = []
        pred_texts = []
        confs = []

        for page in result or []:
            for item in page or []:
                if not item or len(item) < 2:
                    continue
                box = item[0]
                text = str(item[1][0]).strip()
                score = float(item[1][1]) if len(item[1]) > 1 else np.nan

                if not text:
                    continue

                pts = []
                if isinstance(box, (list, tuple)) and len(box) >= 4:
                    for p in box[:4]:
                        try:
                            pts.append([float(p[0]), float(p[1])])
                        except Exception:
                            pts = []
                            break

                if len(pts) != 4:
                    continue

                anns.append({"transcription": text, "points": pts, "difficult": False})
                pred_texts.append(text)
                confs.append(score)

        if len(anns) > 0:
            by_image[str(img_path)] = anns

        pred_doc = " ".join(pred_texts)
        quality_rows.append(
            {
                "image_path": str(img_path),
                "n_boxes": len(anns),
                "mean_conf": float(np.nanmean(confs)) if len(confs) > 0 else np.nan,
                "gt_chars": len(gt_text),
                "pred_chars": len(pred_doc),
                "doc_similarity": doc_similarity(gt_text, pred_doc),
            }
        )

    quality_df = pd.DataFrame(quality_rows)
    return by_image, quality_df

def save_det_labels(by_image: dict, output_file: Path):
    output_file.parent.mkdir(parents=True, exist_ok=True)
    with output_file.open("w", encoding="utf-8") as f:
        for img_path, anns in by_image.items():
            payload = json.dumps(anns, ensure_ascii=False)
            f.write(f"{img_path}\t{payload}\n")

def crop_by_quad(img, pts):
    pts = np.array(pts, dtype=np.float32)
    if pts.shape != (4, 2):
        return None

    s = pts.sum(axis=1)
    diff = np.diff(pts, axis=1).reshape(-1)
    tl = pts[np.argmin(s)]
    br = pts[np.argmax(s)]
    tr = pts[np.argmin(diff)]
    bl = pts[np.argmax(diff)]
    rect = np.array([tl, tr, br, bl], dtype=np.float32)

    width_a = np.linalg.norm(br - bl)
    width_b = np.linalg.norm(tr - tl)
    max_w = int(max(width_a, width_b))
    height_a = np.linalg.norm(tr - br)
    height_b = np.linalg.norm(tl - bl)
    max_h = int(max(height_a, height_b))

    if max_w < 2 or max_h < 2:
        return None

    dst = np.array([[0, 0], [max_w - 1, 0], [max_w - 1, max_h - 1], [0, max_h - 1]], dtype=np.float32)
    m = cv2.getPerspectiveTransform(rect, dst)
    warped = cv2.warpPerspective(img, m, (max_w, max_h), flags=cv2.INTER_CUBIC)

    if warped.shape[0] > warped.shape[1] * 1.5:
        warped = cv2.rotate(warped, cv2.ROTATE_90_CLOCKWISE)
    return warped

def save_rec_dataset(by_image: dict, rec_img_dir: Path, rec_label_file: Path, prefix: str):
    rec_img_dir.mkdir(parents=True, exist_ok=True)
    rec_label_file.parent.mkdir(parents=True, exist_ok=True)

    lines = []
    sample_idx = 0
    for img_path, anns in tqdm(by_image.items(), desc=f"Generate rec crops ({prefix})"):
        img = cv2.imread(img_path)
        if img is None:
            continue
        for ann in anns:
            text = str(ann.get("transcription", "")).strip()
            pts = ann.get("points")
            if not text or pts is None:
                continue
            crop = crop_by_quad(img, pts)
            if crop is None:
                continue
            out_name = f"{prefix}_{sample_idx:07d}.jpg"
            out_path = rec_img_dir / out_name
            cv2.imwrite(str(out_path), crop)
            safe_text = text.replace("\t", " ").replace("\n", " ")
            lines.append(f"{out_path}\t{safe_text}\n")
            sample_idx += 1

    with rec_label_file.open("w", encoding="utf-8") as f:
        f.writelines(lines)

    return len(lines)


In [ ]:
# Construction des donnees: labels documentaires batch_1 -> pseudo labels OCR
batch1_root, batch1_doc_labels, batch1_report = load_batch_document_labels(dataset_path, TRAIN_BATCH_NAME)
train_docs, val_docs = split_document_labels(batch1_doc_labels, TRAIN_VAL_SPLIT, RANDOM_SEED)

batch2_root, batch2_images = list_images_in_batch(dataset_path, UNLABELED_VAL_BATCH_NAME)

print("Batch1 root:", batch1_root)
print("Batch2 root:", batch2_root)
print("Batch1 docs labels (total):", len(batch1_doc_labels))
print("Train docs:", len(train_docs))
print("Val docs (split interne):", len(val_docs))
print("Batch2 images non annotees:", len(batch2_images))

print("\nBatch1 CSV report:")
for r in batch1_report:
    print(f"- {Path(r['csv']).name}: rows={r['rows']}, kept={r['kept_rows']}")

if len(train_docs) == 0 or len(val_docs) == 0:
    raise RuntimeError("Split supervise vide. Verifie les labels batch_1.")

print("\nGeneration des pseudo labels via teacher PaddleOCR...")
train_by_image, train_quality = build_pseudo_annotations_from_teacher(train_docs, use_gpu=USE_GPU, lang="en")
val_by_image, val_quality = build_pseudo_annotations_from_teacher(val_docs, use_gpu=USE_GPU, lang="en")

print("Pseudo labels train images:", len(train_by_image))
print("Pseudo labels val images:", len(val_by_image))
print("Train similarity moyenne:", train_quality['doc_similarity'].mean() if len(train_quality) else float('nan'))
print("Val similarity moyenne:", val_quality['doc_similarity'].mean() if len(val_quality) else float('nan'))

if len(train_by_image) == 0 or len(val_by_image) == 0:
    raise RuntimeError("Aucun pseudo label genere. Verifie le teacher OCR / environnement GPU.")

train_quality_csv = PREP_DIR / "pseudo_quality_train.csv"
val_quality_csv = PREP_DIR / "pseudo_quality_val.csv"
train_quality.to_csv(train_quality_csv, index=False)
val_quality.to_csv(val_quality_csv, index=False)

train_det_label = PREP_DIR / "train_det.txt"
val_det_label = PREP_DIR / "val_det.txt"
save_det_labels(train_by_image, train_det_label)
save_det_labels(val_by_image, val_det_label)

train_rec_label = PREP_DIR / "train_rec.txt"
val_rec_label = PREP_DIR / "val_rec.txt"
train_rec_count = save_rec_dataset(train_by_image, PREP_DIR / "rec_images" / "train", train_rec_label, "train")
val_rec_count = save_rec_dataset(val_by_image, PREP_DIR / "rec_images" / "val", val_rec_label, "val")

batch2_list_file = PREP_DIR / "batch2_unlabeled_images.txt"
batch2_list_file.write_text("\n".join(str(p) for p in batch2_images), encoding="utf-8")

print("\nFichiers generes:")
print("-", train_det_label)
print("-", val_det_label)
print("-", train_rec_label, "(samples=", train_rec_count, ")")
print("-", val_rec_label, "(samples=", val_rec_count, ")")
print("-", train_quality_csv)
print("-", val_quality_csv)
print("-", batch2_list_file)

if train_rec_count == 0 or val_rec_count == 0:
    raise RuntimeError("Le dataset rec est vide apres crop. Verifie la generation des pseudo labels.")


In [ ]:
# Setup PaddleOCR repo (scripts d'entrainement officiels)
PADDLEOCR_REPO = WORK_DIR / "PaddleOCR"
if not PADDLEOCR_REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/PaddlePaddle/PaddleOCR.git", str(PADDLEOCR_REPO)], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PADDLEOCR_REPO / "requirements.txt")], check=True)
print("PaddleOCR repo:", PADDLEOCR_REPO)

In [ ]:
# Selection automatique de configs stables
def pick_first_existing(paths):
    for p in paths:
        if p.exists():
            return p
    return None

det_cfg_candidates = [
    PADDLEOCR_REPO / "configs/det/det_mv3_db.yml",
    PADDLEOCR_REPO / "configs/det/ch_PP-OCRv3/ch_PP-OCRv3_det_student.yml",
    PADDLEOCR_REPO / "configs/det/ch_PP-OCRv4/ch_PP-OCRv4_det_student.yml",
]
rec_cfg_candidates = [
    PADDLEOCR_REPO / "configs/rec/rec_mv3_none_bilstm_ctc.yml",
    PADDLEOCR_REPO / "configs/rec/PP-OCRv3/en_PP-OCRv3_rec.yml",
    PADDLEOCR_REPO / "configs/rec/PP-OCRv4/en_PP-OCRv4_rec.yml",
]

DET_CFG = pick_first_existing(det_cfg_candidates)
REC_CFG = pick_first_existing(rec_cfg_candidates)
if DET_CFG is None or REC_CFG is None:
    raise FileNotFoundError(f"Config introuvable. DET={DET_CFG}, REC={REC_CFG}")

print("DET_CFG:", DET_CFG)
print("REC_CFG:", REC_CFG)

dict_candidates = [
    PADDLEOCR_REPO / "ppocr/utils/en_dict.txt",
    PADDLEOCR_REPO / "ppocr/utils/dict/en_dict.txt",
]
CHAR_DICT = pick_first_existing(dict_candidates)
print("CHAR_DICT:", CHAR_DICT)

In [ ]:
# Suivi des metriques d'entrainement par epoch
plt.style.use("seaborn-v0_8-darkgrid")

METRICS_DIR = RUNS_DIR / "metrics"
METRICS_DIR.mkdir(parents=True, exist_ok=True)

FLOAT_RE = r"[-+]?(?:\d*\.\d+|\d+)(?:[eE][-+]?\d+)?"
EPOCH_RE = re.compile(r"epoch[^0-9]*(\d+)", re.I)
LOSS_RE = re.compile(r"\bloss\b[:=\s]*(" + FLOAT_RE + r")", re.I)
ACC_RE = re.compile(r"\bacc\b[:=\s]*(" + FLOAT_RE + r")", re.I)
NORM_EDIT_RE = re.compile(r"\bnorm_edit_dis\b[:=\s]*(" + FLOAT_RE + r")", re.I)
PRECISION_RE = re.compile(r"\bprecision\b[:=\s]*(" + FLOAT_RE + r")", re.I)
RECALL_RE = re.compile(r"\brecall\b[:=\s]*(" + FLOAT_RE + r")", re.I)
HMEAN_RE = re.compile(r"\bhmean\b[:=\s]*(" + FLOAT_RE + r")", re.I)


def find_first_int_recursive(obj, keys):
    if isinstance(obj, dict):
        for k in keys:
            v = obj.get(k)
            if isinstance(v, int):
                return int(v)
        for v in obj.values():
            found = find_first_int_recursive(v, keys)
            if found is not None:
                return found
    elif isinstance(obj, list):
        for item in obj:
            found = find_first_int_recursive(item, keys)
            if found is not None:
                return found
    return None


def get_batch_size_from_cfg(cfg_path: Path) -> int | None:
    data = yaml.safe_load(cfg_path.read_text(encoding="utf-8"))
    return find_first_int_recursive(data, {"batch_size_per_card", "batch_size", "batch_size_per_gpu"})


def compute_eval_every(train_count: int, batch_size: int | None) -> int:
    bs = max(1, int(batch_size or 1))
    return max(1, math.ceil(train_count / bs))


def summarize_training_log(log_path: Path, task_kind: str) -> pd.DataFrame:
    epochs = defaultdict(lambda: {"train_losses": [], "val_loss": None, "metrics": {}})
    current_epoch = None

    for line in log_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        epoch_match = EPOCH_RE.search(line)
        if epoch_match:
            current_epoch = int(epoch_match.group(1))
        if current_epoch is None:
            continue

        bucket = epochs[current_epoch]
        lower = line.lower()

        loss_match = LOSS_RE.search(line)
        if loss_match:
            loss_val = float(loss_match.group(1))
            if any(token in lower for token in ["eval", "valid", "val"]):
                bucket["val_loss"] = loss_val
            else:
                bucket["train_losses"].append(loss_val)

        if task_kind == "rec":
            for name, rx in [("acc", ACC_RE), ("norm_edit_dis", NORM_EDIT_RE)]:
                m = rx.search(line)
                if m:
                    bucket["metrics"][name] = float(m.group(1))
        else:
            for name, rx in [("precision", PRECISION_RE), ("recall", RECALL_RE), ("hmean", HMEAN_RE)]:
                m = rx.search(line)
                if m:
                    bucket["metrics"][name] = float(m.group(1))

    rows = []
    for epoch in sorted(epochs):
        bucket = epochs[epoch]
        row = {
            "epoch": epoch + 1,
            "epoch_raw": epoch,
            "train_loss": float(np.mean(bucket["train_losses"])) if bucket["train_losses"] else np.nan,
            "val_loss": bucket["val_loss"] if bucket["val_loss"] is not None else np.nan,
            "val_acc": np.nan,
            "val_norm_edit_dis": np.nan,
            "val_cer_proxy": np.nan,
            "val_precision": np.nan,
            "val_recall": np.nan,
            "val_hmean": np.nan,
        }
        if task_kind == "rec":
            row["val_acc"] = bucket["metrics"].get("acc", np.nan)
            row["val_norm_edit_dis"] = bucket["metrics"].get("norm_edit_dis", np.nan)
            if np.isfinite(row["val_norm_edit_dis"]):
                row["val_cer_proxy"] = 1.0 - row["val_norm_edit_dis"]
        else:
            row["val_precision"] = bucket["metrics"].get("precision", np.nan)
            row["val_recall"] = bucket["metrics"].get("recall", np.nan)
            row["val_hmean"] = bucket["metrics"].get("hmean", np.nan)
        rows.append(row)

    return pd.DataFrame(rows)


def plot_training_metrics(df: pd.DataFrame, task_kind: str, title: str, output_png: Path):
    if df.empty:
        print(f"Aucune metrique exploitable pour {title}")
        return None

    fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
    ax_loss, ax_val = axes

    ax_loss.plot(df["epoch"], df["train_loss"], marker="o", label="train loss")
    if df["val_loss"].notna().any():
        ax_loss.plot(df["epoch"], df["val_loss"], marker="o", label="val loss")
    else:
        ax_loss.text(0.02, 0.02, "PaddleOCR ne remonte pas toujours de val_loss dans les logs standards.", transform=ax_loss.transAxes, fontsize=9, alpha=0.75)
    ax_loss.set_title(f"{title} - losses")
    ax_loss.set_xlabel("Epoch")
    ax_loss.set_ylabel("Loss")
    ax_loss.legend()
    ax_loss.grid(True, alpha=0.3)

    if task_kind == "rec":
        if df["val_cer_proxy"].notna().any():
            ax_val.plot(df["epoch"], df["val_cer_proxy"], marker="o", label="CER approx (1 - norm_edit_dis)")
        if df["val_acc"].notna().any():
            ax_val.plot(df["epoch"], df["val_acc"], marker="o", label="val acc")
        if df["val_norm_edit_dis"].notna().any():
            ax_val.plot(df["epoch"], df["val_norm_edit_dis"], marker="o", label="norm_edit_dis")
        ax_val.set_title(f"{title} - validation recognition metrics")
    else:
        if df["val_hmean"].notna().any():
            ax_val.plot(df["epoch"], df["val_hmean"], marker="o", label="hmean")
        if df["val_precision"].notna().any():
            ax_val.plot(df["epoch"], df["val_precision"], marker="o", label="precision")
        if df["val_recall"].notna().any():
            ax_val.plot(df["epoch"], df["val_recall"], marker="o", label="recall")
        ax_val.set_title(f"{title} - validation detection metrics")

    ax_val.set_xlabel("Epoch")
    ax_val.set_ylabel("Metric")
    ax_val.legend()
    ax_val.grid(True, alpha=0.3)

    fig.suptitle(title)
    fig.tight_layout()
    output_png.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_png, dpi=160, bbox_inches="tight")
    display(fig)
    plt.close(fig)
    return output_png


def run_training_with_metrics(cmd, cwd: Path, log_path: Path, task_kind: str, title: str, output_csv: Path, output_png: Path):
    log_path.parent.mkdir(parents=True, exist_ok=True)
    cmd = list(cmd)
    if len(cmd) >= 2 and cmd[1] != "-u":
        cmd.insert(1, "-u")

    print("Running:\n", " ".join(str(x) for x in cmd))
    process = subprocess.Popen(
        cmd,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        universal_newlines=True,
    )

    seen_epochs = set()
    epochs = defaultdict(lambda: {"train_losses": [], "val_loss": None, "metrics": {}})

    def maybe_print_epoch_summary(epoch: int):
        if epoch in seen_epochs:
            return
        bucket = epochs[epoch]
        if task_kind == "rec":
            if not any(name in bucket["metrics"] for name in ["acc", "norm_edit_dis"]):
                return
            train_loss = float(np.mean(bucket["train_losses"])) if bucket["train_losses"] else np.nan
            acc = bucket["metrics"].get("acc", np.nan)
            ned = bucket["metrics"].get("norm_edit_dis", np.nan)
            cer_proxy = 1.0 - ned if np.isfinite(ned) else np.nan
            print(f"[epoch summary] epoch {epoch + 1}: train_loss={train_loss:.4f} | val_acc={acc:.4f} | val_cer_proxy={cer_proxy:.4f}")
        else:
            if not any(name in bucket["metrics"] for name in ["precision", "recall", "hmean"]):
                return
            train_loss = float(np.mean(bucket["train_losses"])) if bucket["train_losses"] else np.nan
            precision = bucket["metrics"].get("precision", np.nan)
            recall = bucket["metrics"].get("recall", np.nan)
            hmean = bucket["metrics"].get("hmean", np.nan)
            print(f"[epoch summary] epoch {epoch + 1}: train_loss={train_loss:.4f} | val_hmean={hmean:.4f} | precision={precision:.4f} | recall={recall:.4f}")
        seen_epochs.add(epoch)

    with log_path.open("w", encoding="utf-8") as log_file:
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
            log_file.write(line)
            log_file.flush()

            epoch_match = EPOCH_RE.search(line)
            if not epoch_match:
                continue
            epoch = int(epoch_match.group(1))
            bucket = epochs[epoch]
            lower = line.lower()

            loss_match = LOSS_RE.search(line)
            if loss_match:
                loss_val = float(loss_match.group(1))
                if any(token in lower for token in ["eval", "valid", "val"]):
                    bucket["val_loss"] = loss_val
                else:
                    bucket["train_losses"].append(loss_val)

            if task_kind == "rec":
                for name, rx in [("acc", ACC_RE), ("norm_edit_dis", NORM_EDIT_RE)]:
                    m = rx.search(line)
                    if m:
                        bucket["metrics"][name] = float(m.group(1))
            else:
                for name, rx in [("precision", PRECISION_RE), ("recall", RECALL_RE), ("hmean", HMEAN_RE)]:
                    m = rx.search(line)
                    if m:
                        bucket["metrics"][name] = float(m.group(1))

            maybe_print_epoch_summary(epoch)

    rc = process.wait()
    if rc != 0:
        raise subprocess.CalledProcessError(rc, cmd)

    df = summarize_training_log(log_path, task_kind)
    output_csv.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_csv, index=False)
    print(f"{title} - resume metriques par epoch:")
    display(df)
    plot_training_metrics(df, task_kind, title, output_png)
    return df


DET_TRAIN_BATCH = int(TRAINING_PROFILE["train_batch_size"])
DET_EVAL_BATCH = int(TRAINING_PROFILE["det_eval_batch_size"])
DET_NUM_WORKERS = int(TRAINING_PROFILE["num_workers"])

REC_TRAIN_BATCH = int(TRAINING_PROFILE["train_batch_size"])
REC_EVAL_BATCH = int(TRAINING_PROFILE["eval_batch_size"])
REC_NUM_WORKERS = int(TRAINING_PROFILE["num_workers"])

USE_AMP = bool(TRAINING_PROFILE["use_amp"] and USE_GPU)
AMP_LEVEL = str(TRAINING_PROFILE["amp_level"])
SCALE_LOSS = float(TRAINING_PROFILE["scale_loss"])

DET_EVAL_EVERY = compute_eval_every(len(train_by_image), DET_TRAIN_BATCH)
REC_EVAL_EVERY = compute_eval_every(train_rec_count, REC_TRAIN_BATCH)

print("DET_TRAIN_BATCH:", DET_TRAIN_BATCH)
print("DET_EVAL_BATCH:", DET_EVAL_BATCH)
print("DET_NUM_WORKERS:", DET_NUM_WORKERS)
print("DET_EVAL_EVERY:", DET_EVAL_EVERY)
print("REC_TRAIN_BATCH:", REC_TRAIN_BATCH)
print("REC_EVAL_BATCH:", REC_EVAL_BATCH)
print("REC_NUM_WORKERS:", REC_NUM_WORKERS)
print("REC_EVAL_EVERY:", REC_EVAL_EVERY)
print("USE_AMP:", USE_AMP, "AMP_LEVEL:", AMP_LEVEL)


In [ ]:
# Entrainement DET (val supervisee = split interne de batch_1)
DET_MODEL_DIR = RUNS_DIR / "det"
DET_MODEL_DIR.mkdir(parents=True, exist_ok=True)

DET_TRAIN_LOG = METRICS_DIR / "det_train.log"
DET_METRICS_CSV = METRICS_DIR / "det_epoch_metrics.csv"
DET_METRICS_PNG = METRICS_DIR / "det_epoch_metrics.png"

cmd_det = [
    sys.executable,
    "-u",
    str(PADDLEOCR_REPO / "tools/train.py"),
    "-c", str(DET_CFG),
    "-o",
    f"Global.use_gpu={str(USE_GPU)}",
    f"Global.epoch_num={DET_EPOCHS}",
    f"Global.save_model_dir={DET_MODEL_DIR}",
    f"Global.save_epoch_step=1",
    f"Global.eval_batch_step=[0,{DET_EVAL_EVERY}]",
    f"Global.use_amp={str(USE_AMP)}",
    f"Global.amp_level={AMP_LEVEL}",
    f"Global.scale_loss={SCALE_LOSS}",
    "Train.dataset.data_dir=/",
    f"Train.dataset.label_file_list=['{train_det_label}']",
    f"Train.loader.batch_size_per_card={DET_TRAIN_BATCH}",
    f"Train.loader.num_workers={DET_NUM_WORKERS}",
    "Eval.dataset.data_dir=/",
    f"Eval.dataset.label_file_list=['{val_det_label}']",
    f"Eval.loader.batch_size_per_card={DET_EVAL_BATCH}",
    f"Eval.loader.num_workers={DET_NUM_WORKERS}",
]

print("Running:\n", " ".join(cmd_det))
det_metrics = run_training_with_metrics(
    cmd_det,
    PADDLEOCR_REPO,
    DET_TRAIN_LOG,
    "det",
    "DET training metrics",
    DET_METRICS_CSV,
    DET_METRICS_PNG,
)
print("DET training termine")

In [ ]:
# Entrainement REC (val supervisee = split interne de batch_1)
REC_MODEL_DIR = RUNS_DIR / "rec"
REC_MODEL_DIR.mkdir(parents=True, exist_ok=True)

REC_TRAIN_LOG = METRICS_DIR / "rec_train.log"
REC_METRICS_CSV = METRICS_DIR / "rec_epoch_metrics.csv"
REC_METRICS_PNG = METRICS_DIR / "rec_epoch_metrics.png"

cmd_rec = [
    sys.executable,
    "-u",
    str(PADDLEOCR_REPO / "tools/train.py"),
    "-c", str(REC_CFG),
    "-o",
    f"Global.use_gpu={str(USE_GPU)}",
    f"Global.epoch_num={REC_EPOCHS}",
    f"Global.save_model_dir={REC_MODEL_DIR}",
    f"Global.save_epoch_step=1",
    f"Global.eval_batch_step=[0,{REC_EVAL_EVERY}]",
    f"Global.use_amp={str(USE_AMP)}",
    f"Global.amp_level={AMP_LEVEL}",
    f"Global.scale_loss={SCALE_LOSS}",
    "Train.dataset.data_dir=/",
    f"Train.dataset.label_file_list=['{train_rec_label}']",
    f"Train.loader.batch_size_per_card={REC_TRAIN_BATCH}",
    f"Train.loader.num_workers={REC_NUM_WORKERS}",
    "Eval.dataset.data_dir=/",
    f"Eval.dataset.label_file_list=['{val_rec_label}']",
    f"Eval.loader.batch_size_per_card={REC_EVAL_BATCH}",
    f"Eval.loader.num_workers={REC_NUM_WORKERS}",
]
if CHAR_DICT is not None:
    cmd_rec += [f"Global.character_dict_path={CHAR_DICT}"]

print("Running:\n", " ".join(cmd_rec))
rec_metrics = run_training_with_metrics(
    cmd_rec,
    PADDLEOCR_REPO,
    REC_TRAIN_LOG,
    "rec",
    "REC training metrics",
    REC_METRICS_CSV,
    REC_METRICS_PNG,
)
print("REC training termine")

In [ ]:
# Export des modeles d'inference
def find_checkpoint_prefix(model_dir: Path):
    for pref in ["best_accuracy", "latest"]:
        if (model_dir / f"{pref}.pdparams").exists():
            return model_dir / pref
    pdparams = sorted(model_dir.glob("*.pdparams"), key=lambda x: x.stat().st_mtime, reverse=True)
    if not pdparams:
        raise FileNotFoundError(f"Aucun checkpoint trouve dans {model_dir}")
    return pdparams[0].with_suffix("")

DET_CKPT = find_checkpoint_prefix(DET_MODEL_DIR)
REC_CKPT = find_checkpoint_prefix(REC_MODEL_DIR)

DET_INFER_DIR = EXPORT_DIR / "det_infer"
REC_INFER_DIR = EXPORT_DIR / "rec_infer"
DET_INFER_DIR.mkdir(parents=True, exist_ok=True)
REC_INFER_DIR.mkdir(parents=True, exist_ok=True)

cmd_export_det = [
    sys.executable, str(PADDLEOCR_REPO / "tools/export_model.py"),
    "-c", str(DET_CFG),
    "-o",
    f"Global.checkpoints={DET_CKPT}",
    f"Global.save_inference_dir={DET_INFER_DIR}",
]
cmd_export_rec = [
    sys.executable, str(PADDLEOCR_REPO / "tools/export_model.py"),
    "-c", str(REC_CFG),
    "-o",
    f"Global.checkpoints={REC_CKPT}",
    f"Global.save_inference_dir={REC_INFER_DIR}",
]
if CHAR_DICT is not None:
    cmd_export_rec += [f"Global.character_dict_path={CHAR_DICT}"]

subprocess.run(cmd_export_det, check=True, cwd=PADDLEOCR_REPO)
subprocess.run(cmd_export_rec, check=True, cwd=PADDLEOCR_REPO)

print("DET_INFER_DIR:", DET_INFER_DIR)
print("REC_INFER_DIR:", REC_INFER_DIR)

In [ ]:
# Inference OCR brute sur un JPG facture
from paddleocr import PaddleOCR

if not TEST_IMAGE.exists():
    raise FileNotFoundError(f"Image de test introuvable: {TEST_IMAGE}")

ocr = PaddleOCR(
    use_angle_cls=True,
    lang="en",
    device=("gpu:0" if USE_GPU else "cpu"),
    det_model_dir=str(DET_INFER_DIR),
    rec_model_dir=str(REC_INFER_DIR),
)

result = ocr.ocr(str(TEST_IMAGE), cls=True)

all_texts = []
all_scores = []
for page in result:
    for item in page:
        if not item or len(item) < 2:
            continue
        text = str(item[1][0]).strip()
        score = float(item[1][1]) if len(item[1]) > 1 else np.nan
        if text:
            all_texts.append(text)
            all_scores.append(score)

print(f"Nombre de chaines extraites: {len(all_texts)}")
if len(all_scores) > 0:
    print(f"Confiance moyenne: {np.nanmean(all_scores):.4f}")

print("\n--- TEXTES EXTRAITS ---")
for i, t in enumerate(all_texts[:200], 1):
    print(f"{i:03d}. {t}")

raw_txt_out = WORK_DIR / "inference_raw_text.txt"
raw_txt_out.write_text("\n".join(all_texts), encoding="utf-8")
print("\nFichier texte sauvegarde:", raw_txt_out)

In [ ]:
# Validation qualitative non annotee sur batch_2
if len(batch2_images) == 0:
    raise RuntimeError("Aucune image trouvee dans batch_2 pour validation qualitative.")

rows = []
max_images = min(BATCH2_MAX_IMAGES, len(batch2_images))
for img_path in tqdm(batch2_images[:max_images], desc="Batch2 qualitative validation"):
    res = ocr.ocr(str(img_path), cls=True)
    texts = []
    confs = []
    for page in res:
        for item in page:
            if not item or len(item) < 2:
                continue
            txt = str(item[1][0]).strip()
            if not txt:
                continue
            score = float(item[1][1]) if len(item[1]) > 1 else np.nan
            texts.append(txt)
            confs.append(score)

    rows.append(
        {
            "image_path": str(img_path),
            "n_texts": len(texts),
            "mean_conf": float(np.nanmean(confs)) if len(confs) > 0 else np.nan,
            "raw_text": " | ".join(texts),
        }
    )

val_df = pd.DataFrame(rows)
val_csv = WORK_DIR / "batch2_unlabeled_validation.csv"
val_df.to_csv(val_csv, index=False)

print("Validation qualitative batch_2 terminee")
print("Images traitees:", len(val_df))
print("Moyenne n_texts:", val_df["n_texts"].mean() if len(val_df) else 0)
print("Moyenne confidence:", val_df["mean_conf"].mean(skipna=True) if len(val_df) else float("nan"))
print("CSV resultat:", val_csv)
val_df.head(10)

## Notes
- Ce dataset n'a pas de bboxes natives dans les CSV (`File Name`, `Json Data`, `OCRed Text`).
- Le notebook genere donc des pseudo-labels bbox+texte via un teacher PaddleOCR sur `batch_1`.
- `batch_2` est utilise pour validation qualitative uniquement (non annote).
- Si l'entrainement est trop long, baisse `DET_EPOCHS` et `REC_EPOCHS` (ex: 10/10).
- Tu peux changer `TEST_IMAGE` pour tester n'importe quel JPG de facture.
- Le notebook adapte automatiquement batch/workers/AMP selon la VRAM GPU detectee (profil dynamique).
- Le palier `gradient_accumulation_steps_requested` est informatif; il n'est pas applique nativement par `tools/train.py` dans cette version de PaddleOCR.